In [ ]:
import pandas as pd
import numpy as np

import torch
import torch.nn as nn
from catboost import CatBoostRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
cd drive/MyDrive/

/content/drive/MyDrive


In [ ]:
train = pd.read_csv('traffic_V2.csv')
test = pd.read_csv('test_traffic_V2.csv')

print(f"학습 데이터 크기: {train.shape}")
print(f"테스트 데이터 크기: {test.shape}")

학습 데이터 크기: (250000, 94)
테스트 데이터 크기: (50000, 93)


In [ ]:
TARGET = 'avg_delay_minutes_next_30m'
ID_COLS = ['ID', 'layout_id', 'scenario_id']

feature_cols = [c for c in train.columns if c not in ID_COLS + [TARGET]]
print(f"피처 수: {len(feature_cols)}")

피처 수: 90


In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
oof_preds = np.zeros(len(train))
test_preds = np.zeros(len(test))

for fold, (tr_idx, val_idx) in enumerate(kf.split(train)):
    print(f"── Fold {fold + 1} ──")
    X_tr = train.loc[tr_idx, feature_cols]
    y_tr = train.loc[tr_idx, TARGET]
    X_val = train.loc[val_idx, feature_cols]
    y_val = train.loc[val_idx, TARGET]

    model = CatBoostRegressor(
        iterations=1000,
        learning_rate=0.05,
        depth=7,
        l2_leaf_reg=3,
        loss_function='MAE',
        eval_metric='MAE',
        random_seed=42,
        task_type='CPU',   # GPU 사용 (Colab이면 가능)
        verbose=100
    )

    model.fit(
        X_tr, y_tr,
        eval_set=(X_val, y_val),
        early_stopping_rounds=50,
        use_best_model=True
    )

    oof_preds[val_idx] = model.predict(X_val)
    test_preds += model.predict(test[feature_cols]) / 5

── Fold 1 ──
0:	learn: 14.1934710	test: 14.2479834	best: 14.2479834 (0)	total: 334ms	remaining: 5m 34s
100:	learn: 9.3787540	test: 9.3930362	best: 9.3930362 (100)	total: 16s	remaining: 2m 22s
200:	learn: 9.2710029	test: 9.3175326	best: 9.3175326 (200)	total: 34.4s	remaining: 2m 16s
300:	learn: 9.1642479	test: 9.2446928	best: 9.2446928 (300)	total: 49.9s	remaining: 1m 55s
400:	learn: 9.0570790	test: 9.1768137	best: 9.1768137 (400)	total: 1m 5s	remaining: 1m 37s
500:	learn: 8.9671198	test: 9.1212254	best: 9.1212254 (500)	total: 1m 21s	remaining: 1m 21s
600:	learn: 8.8849711	test: 9.0739206	best: 9.0739206 (600)	total: 1m 38s	remaining: 1m 5s
700:	learn: 8.8113342	test: 9.0327511	best: 9.0327511 (700)	total: 1m 54s	remaining: 48.7s
800:	learn: 8.7477196	test: 8.9999032	best: 8.9999032 (800)	total: 2m 10s	remaining: 32.5s
900:	learn: 8.6873011	test: 8.9681013	best: 8.9681013 (900)	total: 2m 25s	remaining: 16s
999:	learn: 8.6294367	test: 8.9387555	best: 8.9387555 (999)	total: 2m 41s	remaini

In [ ]:
oof_mae = mean_absolute_error(train[TARGET], oof_preds)
print(f"OOF MAE: {oof_mae:.4f}")

OOF MAE: 8.9698


In [ ]:
submission = pd.DataFrame({'ID': test['ID'], TARGET: test_preds})
submission.to_csv('./submission_V16.csv', index=False)
print("submission.csv 저장 완료.")

submission.csv 저장 완료.
